<a href="https://colab.research.google.com/github/yamamoto-yuuichi/Streamlit-x-Knowledge-Graph/blob/main/Streamlit%E3%81%A7%E3%83%8A%E3%83%AC%E3%83%83%E3%82%B8%E3%82%B0%E3%83%A9%E3%83%95%E3%81%AB%E6%8C%91%E6%88%A6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

【GraphRAG】非エンジニアでも「ポチポチ」と知識を繋げる！Streamlitでナレッジグラフ編集ツールを自作してみた

url
https://zenn.dev/lluminai_tech/articles/887299dcd1d90d

In [ ]:
!pip install streamlit streamlit-agraph networkx
# Webアプリ作成・グラフ表示・グラフ計算の3つのライブラリを一括インストールする

In [ ]:
!pip install streamlit streamlit-agraph networkx pyngrok -q
# Webアプリ・グラフ表示・グラフ計算・外部公開の4つのライブラリを、ログを省略しながら一括インストールする

In [ ]:
%%writefile app.py
import streamlit as st  # WebアプリをPythonだけで作れるStreamlitを読み込む
from streamlit_agraph import agraph, Node, Edge, Config  # グラフ図を表示するための部品を読み込む
import networkx as nx  # グラフ構造（ノードと矢印の関係）を管理するライブラリを読み込む

if 'graph' not in st.session_state:  # セッション（画面の状態）にグラフデータがなければ
    st.session_state['graph'] = nx.DiGraph()  # 空の有向グラフ（矢印に向きがあるグラフ）を作って保存する
if 'source_node' not in st.session_state:  # 始点ノードの情報がセッションになければ
    st.session_state['source_node'] = None  # 始点を「未選択」として初期化する
if 'target_node' not in st.session_state:  # 終点ノードの情報がセッションになければ
    st.session_state['target_node'] = None  # 終点を「未選択」として初期化する

st.sidebar.header("📦 ノード（要素）の追加")  # サイドバーに見出しを表示する
new_node = st.sidebar.text_input("新しいノード名を入力", placeholder="新しいノード名")  # サイドバーにテキスト入力欄を表示し、入力値を変数に入れる

if st.sidebar.button("➕ ノードを追加"):  # 「ノードを追加」ボタンが押されたら
    if new_node:  # テキスト入力欄に何か入力されていれば
        if not st.session_state['graph'].has_node(new_node):  # 同じ名前のノードがまだグラフに存在しなければ
            st.session_state['graph'].add_node(new_node)  # 入力された名前でノードをグラフに追加する
            st.sidebar.success(f"追加しました: {new_node}")  # 追加成功のメッセージをサイドバーに表示する
        else:  # 同じ名前のノードがすでに存在する場合
            st.sidebar.warning("そのノードは既に存在します。")  # 警告メッセージをサイドバーに表示する
    else:  # 入力欄が空の場合
        st.sidebar.warning("ノード名を入力してください。")  # 入力を促す警告メッセージを表示する

st.sidebar.divider()  # サイドバーに区切り線を表示する
st.sidebar.markdown(f"**現在の要素数:** {st.session_state['graph'].number_of_nodes()}")  # 現在のノード数をサイドバーに表示する
st.sidebar.markdown(f"**現在の関係数:** {st.session_state['graph'].number_of_edges()}")  # 現在のエッジ（矢印）数をサイドバーに表示する

if st.sidebar.button("🗑️ 全データをリセット", type="primary"):  # 「全データをリセット」ボタンが押されたら
    st.session_state['graph'].clear()  # グラフのノードとエッジをすべて削除する
    st.session_state['source_node'] = None  # 始点の選択状態をリセットする
    st.session_state['target_node'] = None  # 終点の選択状態をリセットする
    st.rerun()  # 画面を最初から再描画する

st.title("🕸️ GraphRAG Visual Editor")  # ページのタイトルを表示する
st.markdown("ノードをクリックして選択し、関係性を定義してください。")  # 操作説明のテキストを表示する

col_graph, col_control = st.columns([3, 1])  # 画面を3:1の割合で左右2列に分ける

with col_graph:  # 左側の広い列の中に以下を配置する
    nodes = []  # グラフ表示用のノードリストを空で初期化する
    edges = []  # グラフ表示用のエッジリストを空で初期化する
    for n in st.session_state['graph'].nodes():  # グラフ内の全ノードを一つずつ取り出す
        color = "#F7A7A6"  # ノードの色をデフォルト（薄いピンク）に設定する
        if n == st.session_state['source_node']:  # そのノードが始点として選ばれていれば
            color = "#5D5CDE"  # 色を青紫に変える
        elif n == st.session_state['target_node']:  # そのノードが終点として選ばれていれば
            color = "#4CAF50"  # 色を緑に変える
        nodes.append(Node(id=n, label=n, size=25, color=color))  # 色・サイズを設定したノードをリストに追加する
    for u, v, d in st.session_state['graph'].edges(data=True):  # グラフ内の全エッジを始点・終点・属性付きで取り出す
        edges.append(Edge(source=u, target=v, label=d.get('relation', ''), type="CURVE_SMOOTH"))  # 関係名ラベル付きの曲線エッジをリストに追加する
    config = Config(width="100%", height=500, directed=True, nodeHighlightBehavior=True,
                    highlightColor="#F7A7A6", collapsible=False, physics=True, hierarchical=False)  # グラフ表示の設定（サイズ・物理演算・ハイライト色など）をまとめる
    selected_node_id = agraph(nodes=nodes, edges=edges, config=config)  # グラフを画面に描画し、クリックされたノードのIDを受け取る

with col_control:  # 右側の狭い列の中に以下を配置する
    st.subheader("🛠️ 接続操作")  # 「接続操作」という小見出しを表示する
    if selected_node_id:  # グラフ上でノードがクリックされていれば
        st.info(f"選択中: **{selected_node_id}**")  # 選択中のノード名を情報ボックスで表示する
        c1, c2 = st.columns(2)  # コントロール列をさらに2列に分ける
        with c1:  # 左の列に
            if st.button("始点に設定"):  # 「始点に設定」ボタンが押されたら
                st.session_state['source_node'] = selected_node_id  # 選択中のノードを始点として保存する
                st.rerun()  # 画面を再描画する
        with c2:  # 右の列に
            if st.button("終点に設定"):  # 「終点に設定」ボタンが押されたら
                st.session_state['target_node'] = selected_node_id  # 選択中のノードを終点として保存する
                st.rerun()  # 画面を再描画する
    else:  # ノードが何もクリックされていなければ
        st.write("👈 グラフの丸をクリックしてください")  # 操作を促すメッセージを表示する
    st.divider()  # 区切り線を表示する
    src = st.session_state['source_node']  # 現在の始点ノードを変数に取り出す
    tgt = st.session_state['target_node']  # 現在の終点ノードを変数に取り出す
    st.write(f"**始点 (From):** {src if src else '未選択'}")  # 始点の状態を表示する（未選択なら「未選択」と出す）
    st.write(f"**終点 (To):** {tgt if tgt else '未選択'}")  # 終点の状態を表示する（未選択なら「未選択」と出す）
    if src and tgt:  # 始点と終点の両方が選ばれていれば
        if src == tgt:  # 始点と終点が同じノードなら
            st.warning("自分自身には接続できません")  # 自己ループを禁止する警告を表示する
        else:  # 始点と終点が別のノードなら
            relation_label = st.text_input("関係名 (エッジ名)", key="rel_input")  # 関係名の入力欄を表示し、入力値を変数に入れる
            if st.button("🔗 接続する"):  # 「接続する」ボタンが押されたら
                if relation_label:  # 関係名が入力されていれば
                    st.session_state['graph'].add_edge(src, tgt, relation=relation_label)  # 始点から終点へ関係名付きのエッジをグラフに追加する
                    st.session_state['source_node'] = None  # 始点の選択をリセットする
                    st.session_state['target_node'] = None  # 終点の選択をリセットする
                    st.success(f"接続しました: {src} -> {tgt}")  # 接続成功のメッセージを表示する
                    st.rerun()  # 画面を再描画する
                else:  # 関係名が空のままなら
                    st.error("関係名を入力してください")  # エラーメッセージを表示する
    if src or tgt:  # 始点か終点のどちらかが選ばれていれば
        if st.button("選択クリア"):  # 「選択クリア」ボタンが押されたら
            st.session_state['source_node'] = None  # 始点の選択をリセットする
            st.session_state['target_node'] = None  # 終点の選択をリセットする
            st.rerun()  # 画面を再描画する

st.divider()  # メインエリアに区切り線を表示する
with st.expander("📊 生成されたグラフデータ (JSON形式)"):  # クリックで開閉できる折りたたみエリアを作る
    graph_data = nx.node_link_data(st.session_state['graph'])  # グラフのデータをJSON化できる形式（辞書）に変換する
    st.json(graph_data)  # 変換したグラフデータをJSON形式で画面に表示する

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

In [ ]:
import os

# 既存ファイルを削除
if os.path.exists("/usr/local/bin/cloudflared"):
    os.remove("/usr/local/bin/cloudflared")

# 再ダウンロード
result = subprocess.run([
    "wget", "-q", "-L",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], capture_output=True, text=True)


In [ ]:
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
result = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)

In [ ]:
import subprocess
import time
import re

# cloudflaredをインストール
subprocess.run([
    "wget", "-q", "-L",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Streamlitをバックグラウンドで起動
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(5)

# cloudflaredでトンネルを開く
cf = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# URLが表示されるまで待つ
print("URLを取得中...")
for line in cf.stdout:
    text = line.decode("utf-8")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", text)
    if match:
        print("✅ アクセスURL:", match.group())
        break